### Functional Manner to Compute Results Table - Forecast Error Variance Decomposition (0bp cut event)

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR

In [2]:
def prep_event_data(cme_df, pm_df, kalshi_df=None):
    df_cme = cme_df[["time", "prob_no_change"]].copy()
    df_cme["time"] = pd.to_datetime(df_cme["time"])
    df_cme = df_cme.rename(columns={"time": "timestamp"}).set_index("timestamp")
    cme_min = df_cme.resample("1Min").last()

    # Standardize Polymarket
    df_pm = pm_df[["datetime", "yes_price"]].copy()
    df_pm["datetime"] = pd.to_datetime(df_pm["datetime"])
    df_pm = df_pm.rename(columns={"datetime": "timestamp", "yes_price": "yes_price_PM"}).set_index("timestamp")
    pm_min = df_pm.resample("1Min").last().shift(1)

    # Merge Base Data
    merged_df = pd.merge(cme_min, pm_min, left_index=True, right_index=True, how="outer")

    # Add Kalshi
    if kalshi_df is not None:
        df_k = kalshi_df[["created_time", "yes_price"]].copy()
        df_k["created_time"] = pd.to_datetime(df_k["created_time"], format='mixed')
        df_k = df_k.rename(columns={"created_time": "timestamp", "yes_price": "yes_price_KAL"}).set_index("timestamp")
        kalshi_min = df_k.resample("1Min").last().shift(1)
        merged_df = pd.merge(merged_df, kalshi_min, left_index=True, right_index=True, how="outer")

    # Clean & Fill
    merged_df = merged_df.sort_index().ffill().dropna()

    # Active Zone Filtering
    window_mins = 3 * 24 * 60
    rolling_var = merged_df["prob_no_change"].rolling(window=window_mins, min_periods=window_mins).var()
    is_active = rolling_var > 1e-6
    
    if is_active.any():
        first_active_t = merged_df[is_active].index.min()
        start_t = max(merged_df.index.min(), first_active_t - pd.Timedelta(minutes=window_mins))
        merged_df = merged_df.loc[start_t:]
    
    return merged_df

In [3]:
def calculate_fevd_for_pair(df_clean, source_market, target_market, event_date, contract_type):
    # Mapping column names
    col_map = {"CME": "prob_no_change", "PM": "yes_price_PM", "KAL": "yes_price_KAL"}
    
    # Prepare data
    df_pair = df_clean[[col_map[source_market], col_map[target_market]]].diff().dropna()
    
    # Fit VAR
    model = VAR(df_pair)
    results = model.fit(maxlags=61, ic="aic")
    fevd = results.fevd(61)
    
    results_list = []
    lags = [1, 5, 30, 60]
    
    # In fevd.decomp, the order corresponds to df_pair columns
    # To get source -> target, we look at the target's column (index 1) and source's shock (index 0)
    for lag in lags:
        val = fevd.decomp[1][lag, 0] 
        results_list.append({
            "event_date": event_date,
            "contract": contract_type,
            "source_market": source_market,
            "target_market": target_market,
            "lag_minute": lag,
            "FEVD": val
        })
    return results_list

### Forecast Error Variance Decomposition Results for All Events (0bp cut Market)

In [5]:
event_config = {
    "2024-01-31": {"month_code": "JAN"},
    "2024-03-20": {"month_code": "MAR"},
    "2024-06-12": {"month_code": "JUN"},
    "2024-07-31": {"month_code": "JUL"},
    "2024-11-07": {"month_code": "NOV"},
    "2024-12-18": {"month_code": "DEC"},
    "2025-01-29": {"month_code": "JAN"},
    "2025-03-19": {"month_code": "MAR"},
    "2025-05-07": {"month_code": "MAY"},
    "2025-06-18": {"month_code": "JUN"},
    "2025-07-30": {"month_code": "JUL"},
    "2025-09-17": {"month_code": "SEP"},
    "2025-10-29": {"month_code": "OCT"},
    "2025-12-10": {"month_code": "DEC"},
    "2026-01-28": {"month_code": "JAN"},
    "2026-03-18": {"month_code": "MAR"},
    "2026-04-29": {"month_code": "APR"}
}

market_pairs = [("CME", "PM"), ("CME", "KAL"), ("PM", "KAL")]
all_fevd_results = []

# Execution Loop
for slug, config in event_config.items():
    month = config["month_code"]
    year = slug.split('-')[0]
    
    print(f"Processing: {slug}")
    
    try:
        # Load Files
        cme_raw = pd.read_csv(f"../../data/processed/CME_implied_probabilities/CME_IMP_{month}_{year}.csv")
        pm_raw = pd.read_csv(f"../../data/raw/Polymarket/data_0bp_cut/PM_{month}_{year}.csv")
        kal_raw = pd.read_csv(f"../../data/raw/Kalshi/data_0bp_cut/kalshi_{month}_{year}.csv")
        
        # Clean
        df_clean = prep_event_data(cme_raw, pm_raw, kal_raw)
        
        # Run pairs (both directions)
        for mkt_a, mkt_b in market_pairs:
            all_fevd_results.extend(calculate_fevd_for_pair(df_clean, mkt_a, mkt_b, slug, "0bp"))
            all_fevd_results.extend(calculate_fevd_for_pair(df_clean, mkt_b, mkt_a, slug, "0bp"))
            
    except Exception as e:
        print(f"Error on {slug}: {e}")

# Final DataFrame
fevd_df = pd.DataFrame(all_fevd_results)

Processing: 2024-01-31
Processing: 2024-03-20
Processing: 2024-06-12
Processing: 2024-07-31
Processing: 2024-11-07
Processing: 2024-12-18
Processing: 2025-01-29
Processing: 2025-03-19
Processing: 2025-05-07
Processing: 2025-06-18
Processing: 2025-07-30
Processing: 2025-09-17
Processing: 2025-10-29
Processing: 2025-12-10
Processing: 2026-01-28
Processing: 2026-03-18
Processing: 2026-04-29


In [6]:
# Viewing the results as dataframe
fevd_df

,event_date,contract,source_market,target_market,lag_minute,FEVD
0,2024-01-31,0bp,CME,PM,1,0.000009
1,2024-01-31,0bp,CME,PM,5,0.000839
2,2024-01-31,0bp,CME,PM,30,0.004766
3,2024-01-31,0bp,CME,PM,60,0.013682
4,2024-01-31,0bp,PM,CME,1,0.000002
...,...,...,...,...,...,...
403,2026-04-29,0bp,PM,KAL,60,0.000942
404,2026-04-29,0bp,KAL,PM,1,0.001744
405,2026-04-29,0bp,KAL,PM,5,0.001764
406,2026-04-29,0bp,KAL,PM,30,0.005081


In [ ]:
# Saving the dataframe in the defined directory 
fevd_df.to_csv("../../results/forecast_error_variance_decomposition/forecast_error_variance_decomposition_results_0bp_cut.csv", index = False)